# Suite2p Notebook

## Imports and Config

In [25]:
import os
import numpy as np
import h5py

suite2p_root = os.path.join("libs", "suite2p")

movie_path = os.path.join("libs", "ciatah", "data", "twoPhoton", "2017_04_16_p485_m487_runningWheel02", "concat_recording_20140807_102507.h5")
output_dir = os.path.join("data", "suite2p_output")
trace_dir = os.path.join("data", "suite2p_traces")
h5_key = "1"

## CIAtah Movie Stats

In [26]:
with h5py.File(movie_path, "r") as movie_file:
    print("movie path:", movie_path)
    print("root keys:", list(movie_file.keys()))
    movie = movie_file[h5_key]
    print("shape:", movie.shape)
    print("dtype:", movie.dtype)

movie path: libs/ciatah/data/twoPhoton/2017_04_16_p485_m487_runningWheel02/concat_recording_20140807_102507.h5
root keys: ['1', 'movie']
shape: (3000, 201, 201)
dtype: int16


## Suite2p Setup

In [27]:
from suite2p import run_s2p
from suite2p.parameters import default_db, default_settings

frame_rate = 30.0
tau = 0.7
diameter = 12.0
neuropil_coeff = 0.7
baseline_percentile = 20.0
keep_only_cell_rois = True

os.makedirs(output_dir, exist_ok=True)
suite2p_output_dir = os.path.join(output_dir, "suite2p")

db = default_db()
db.update(
    {
        "h5py": movie_path,
        "h5py_key": h5_key,
        "input_format": "h5",
        "data_path": [os.path.dirname(movie_path)],
        "save_path0": output_dir,
        "save_folder": "suite2p",
        "nplanes": 1,
        "nchannels": 1,
    }
)

settings = default_settings()
settings.update(
    {
        "fs": frame_rate,
        "tau": tau,
        "diameter": diameter,
        "save_mat": True,
        "delete_bin": False,
    }
)

## Run Suite2p

In [28]:
run_s2p(db=db, settings=settings)

['data/suite2p_output/suite2p/plane0/db.npy']

## Compute dF/F traces for CASCADE and OASIS

In [29]:
# Convert fluorescence traces to dF/F
def compute_dff(traces, percentile):
    # Use a low percentile baseline to approximate "inactive" fluorescence level `F`
    F = np.percentile(traces, percentile, axis=1, keepdims=True)
    # Avoid division by zero
    F[np.abs(F) < 1e-6] = 1.0
    dF = traces - F
    return dF / np.abs(F)


plane_dir = os.path.join(suite2p_output_dir, "plane0")
fluorescence = np.load(os.path.join(plane_dir, "F.npy"))
neuropil = np.load(os.path.join(plane_dir, "Fneu.npy"))
iscell = np.load(os.path.join(plane_dir, "iscell.npy"))

# Subtract a fixed fraction of the neuropil trace before computing dF/F
corrected = fluorescence - neuropil_coeff * neuropil
traces = compute_dff(corrected, baseline_percentile)

# Default to keeping only Regions of Interest (ROIs) that Suite2p marked as cells
if keep_only_cell_rois:
    keep_indices = np.flatnonzero(iscell[:, 0].astype(bool))
else:
    keep_indices = np.arange(traces.shape[0])

print(f"Suite2p found {keep_indices.size} cells and {traces.shape[0] - keep_indices.size} non-cell ROIs")

selected_traces = traces[keep_indices].astype(np.float32)

Suite2p found 39 cells and 10 non-cell ROIs


## Save dF/F traces

In [30]:
os.makedirs(trace_dir, exist_ok=True)

np.save(os.path.join(trace_dir, "suite2p_dff_traces.npy"), selected_traces)
np.savetxt(os.path.join(trace_dir, "suite2p_dff_traces.csv"), selected_traces, delimiter=",")

np.savetxt(
    os.path.join(trace_dir, "suite2p_selected_rois.csv"),
    np.column_stack((keep_indices, iscell[keep_indices, 0], iscell[keep_indices, 1])),
    delimiter=",",
    header="roi_index,iscell,cell_probability",
    comments="",
)

print(f"Saved CASCADE/OASIS dF/F traces to {trace_dir}")
print(f"Trace shape: {selected_traces.shape[0]} ROIs x {selected_traces.shape[1]} timepoints")

Saved CASCADE/OASIS dF/F traces to data/suite2p_traces
Trace shape: 39 ROIs x 3000 timepoints
